# Establishing Connectivity - the Engine


In [7]:
from sqlalchemy import create_engine

In [8]:
engine = create_engine("sqlite+pysqlite:///:memory:", echo=True)

# Working with Transactions and the DBAPI


## Getting a Connection


In [9]:
from sqlalchemy import text

In [10]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT 'Hello World!'"))
    print(result.all())

2025-04-22 17:07:14,614 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:14,615 INFO sqlalchemy.engine.Engine SELECT 'Hello World!'
2025-04-22 17:07:14,615 INFO sqlalchemy.engine.Engine [generated in 0.00131s] ()
[('Hello World!',)]
2025-04-22 17:07:14,616 INFO sqlalchemy.engine.Engine ROLLBACK


## Committing Changes


### commit as you go


Insert some data and then commit the transaction using the Connection.commit() method, inside the block where we have the Connection object


In [11]:
with engine.connect() as conn:
    conn.execute(text("CREATE TABLE some_table (x int, y int)"))
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"),
        [{"x": 1, "y": 2}, {"x": 3, "y": 4}],
    )
    conn.commit()

2025-04-22 17:07:14,624 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:14,625 INFO sqlalchemy.engine.Engine CREATE TABLE some_table (x int, y int)
2025-04-22 17:07:14,625 INFO sqlalchemy.engine.Engine [generated in 0.00159s] ()
2025-04-22 17:07:14,626 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2025-04-22 17:07:14,627 INFO sqlalchemy.engine.Engine [generated in 0.00066s] [(1, 2), (3, 4)]
2025-04-22 17:07:14,627 INFO sqlalchemy.engine.Engine COMMIT


### begin once


This method will manage the scope of the Connection and also enclose everything inside of a transaction with either a COMMIT at the end if the block was successful, or a ROLLBACK if an exception was raised.


In [12]:
with engine.begin() as conn:
    result = conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"),
        [{"x": 5, "y": 6}, {"x": 7, "y": 8}],
    )
    print(result)

2025-04-22 17:08:23,904 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:08:23,905 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2025-04-22 17:08:23,906 INFO sqlalchemy.engine.Engine [cached since 69.28s ago] [(5, 6), (7, 8)]
2025-04-22 17:08:23,906 INFO sqlalchemy.engine.Engine COMMIT


## Basics of Statement Execution


### Fetching Rows


In [13]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x,y FROM some_table"))
    for row in result:
        print(f"{row.x=}, {row.y=}")

2025-04-22 17:07:14,665 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:14,670 INFO sqlalchemy.engine.Engine SELECT x,y FROM some_table
2025-04-22 17:07:14,671 INFO sqlalchemy.engine.Engine [generated in 0.00590s] ()
row.x=1, row.y=2
row.x=3, row.y=4
row.x=5, row.y=6
row.x=7, row.y=8
2025-04-22 17:07:14,671 INFO sqlalchemy.engine.Engine ROLLBACK


#### Variety of ways to access rows:


1. **Tuple Assignment** - This is the most Python-idiomatic style, which is to assign variables to each row positionally as they are received:


In [14]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x,y FROM some_table"))
    for x, y in result:
        print(f"{x=}, {y=}")

2025-04-22 17:07:14,684 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:14,691 INFO sqlalchemy.engine.Engine SELECT x,y FROM some_table
2025-04-22 17:07:14,692 INFO sqlalchemy.engine.Engine [cached since 0.02686s ago] ()
x=1, y=2
x=3, y=4
x=5, y=6
x=7, y=8
2025-04-22 17:07:14,692 INFO sqlalchemy.engine.Engine ROLLBACK


2. **Integer Index** - Tuples are Python sequences, so regular integer access is available too:


In [15]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x,y FROM some_table"))
    for row in result:
        print(f"{row[0]=}, {row[1]=}")

2025-04-22 17:07:14,705 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:14,707 INFO sqlalchemy.engine.Engine SELECT x,y FROM some_table
2025-04-22 17:07:14,713 INFO sqlalchemy.engine.Engine [cached since 0.04857s ago] ()
row[0]=1, row[1]=2
row[0]=3, row[1]=4
row[0]=5, row[1]=6
row[0]=7, row[1]=8
2025-04-22 17:07:14,714 INFO sqlalchemy.engine.Engine ROLLBACK


3. **Attribute Name** - As these are Python named tuples, the tuples have dynamic attribute names matching the names of each column. These names are normally the names that the SQL statement assigns to the columns in each row. While they are usually fairly predictable and can also be controlled by labels, in less defined cases they may be subject to database-specific behaviors:


In [16]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x,y FROM some_table"))
    for row in result:
        y = row.y
        print(f"{row.x=}, {y=}")

2025-04-22 17:07:14,723 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:14,724 INFO sqlalchemy.engine.Engine SELECT x,y FROM some_table
2025-04-22 17:07:14,724 INFO sqlalchemy.engine.Engine [cached since 0.05947s ago] ()
row.x=1, y=2
row.x=3, y=4
row.x=5, y=6
row.x=7, y=8
2025-04-22 17:07:14,725 INFO sqlalchemy.engine.Engine ROLLBACK


4. **Mapping Access** - To receive rows as Python mapping objects, which is essentially a read-only version of Python’s interface to the common dict object, the Result may be transformed into a MappingResult object using the Result.mappings() modifier; this is a result object that yields dictionary-like RowMapping objects rather than Row objects:


In [17]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x,y FROM some_table"))
    for dict_row in result.mappings():
        x = dict_row["x"]
        y = dict_row["y"]
        print(f"{x=}, {y=}")

2025-04-22 17:07:14,732 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:14,733 INFO sqlalchemy.engine.Engine SELECT x,y FROM some_table
2025-04-22 17:07:14,733 INFO sqlalchemy.engine.Engine [cached since 0.06855s ago] ()
x=1, y=2
x=3, y=4
x=5, y=6
x=7, y=8
2025-04-22 17:07:14,734 INFO sqlalchemy.engine.Engine ROLLBACK


In [18]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x,y FROM some_table")).all()
    print(type(result))
    print(result)
    for x,y in result:
        print(x,y)

2025-04-22 17:07:14,747 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:14,747 INFO sqlalchemy.engine.Engine SELECT x,y FROM some_table
2025-04-22 17:07:14,748 INFO sqlalchemy.engine.Engine [cached since 0.08308s ago] ()
<class 'list'>
[(1, 2), (3, 4), (5, 6), (7, 8)]
1 2
3 4
5 6
7 8
2025-04-22 17:07:14,749 INFO sqlalchemy.engine.Engine ROLLBACK


### Sending Parameters


In [19]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x,y FROM some_table WHERE y > :y"), {"y": 2})
    for row in result:
        print(f"{row.x=}, {row.y=}")

2025-04-22 17:07:14,754 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:14,755 INFO sqlalchemy.engine.Engine SELECT x,y FROM some_table WHERE y > ?
2025-04-22 17:07:14,755 INFO sqlalchemy.engine.Engine [generated in 0.00109s] (2,)
row.x=3, row.y=4
row.x=5, row.y=6
row.x=7, row.y=8
2025-04-22 17:07:14,755 INFO sqlalchemy.engine.Engine ROLLBACK


### Sending Multiple Parameters


In [20]:
with engine.connect() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"),
        [{"x": 9, "y": 10}, {"x": 11, "y": 12}],
    )
    conn.commit()

2025-04-22 17:07:14,761 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:14,762 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2025-04-22 17:07:14,762 INFO sqlalchemy.engine.Engine [cached since 0.1364s ago] [(9, 10), (11, 12)]
2025-04-22 17:07:14,763 INFO sqlalchemy.engine.Engine COMMIT


## Executing with an ORM Session


In [21]:
from sqlalchemy.orm import Session

In [22]:
stmt = text("SELECT x,y FROM some_table WHERE y > :y ORDER BY x, y")
with Session(engine) as session:
    result = session.execute(stmt, {"y": 4})
    for row in result:
        print(f"{row.x=}, {row.y=}")

2025-04-22 17:07:14,826 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:14,827 INFO sqlalchemy.engine.Engine SELECT x,y FROM some_table WHERE y > ? ORDER BY x, y
2025-04-22 17:07:14,827 INFO sqlalchemy.engine.Engine [generated in 0.00062s] (4,)
row.x=5, row.y=6
row.x=7, row.y=8
row.x=9, row.y=10
row.x=11, row.y=12
2025-04-22 17:07:14,828 INFO sqlalchemy.engine.Engine ROLLBACK


In [23]:
with Session(engine) as session:
    session.execute(
        text("UPDATE some_table SET y=:y WHERE x=:x"),
        [{"x": 4, "y": 4}, {"x": 5, "y": 5}],
    )
    session.commit()

2025-04-22 17:07:14,834 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:14,835 INFO sqlalchemy.engine.Engine UPDATE some_table SET y=? WHERE x=?
2025-04-22 17:07:14,836 INFO sqlalchemy.engine.Engine [generated in 0.00072s] [(4, 4), (5, 5)]
2025-04-22 17:07:14,836 INFO sqlalchemy.engine.Engine COMMIT


# Working with Database Metadata


## Setting up MetaData with Table objects


In [24]:
from sqlalchemy import MetaData

In [25]:
metadata_obj = MetaData()

In [26]:
from sqlalchemy import Table, Column, Integer, String

In [27]:
user_table = Table(
    "user_account",
    metadata_obj,
    Column("id", Integer, primary_key=True),
    Column("name", String(30)),
    Column("fullname", String),
)

In [28]:
user_table.c.name

Column('name', String(length=30), table=<user_account>)

In [29]:
user_table.c.keys()

['id', 'name', 'fullname']

### Declaring Simple Constraints


In [30]:
user_table.primary_key

PrimaryKeyConstraint(Column('id', Integer(), table=<user_account>, primary_key=True, nullable=False))

In [31]:
from sqlalchemy import ForeignKey

In [32]:
address_table = Table(
    "address",
    metadata_obj,
    Column("id", Integer, primary_key=True),
    Column("user_id", ForeignKey("user_account.id"), nullable=False),
    Column("email_address", String, nullable=False),
)

### Emitting DDL to the Database


In [33]:
metadata_obj.create_all(engine)

2025-04-22 17:07:14,899 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:14,900 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("user_account")
2025-04-22 17:07:14,900 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-04-22 17:07:14,900 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("user_account")
2025-04-22 17:07:14,901 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-04-22 17:07:14,901 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("address")
2025-04-22 17:07:14,902 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-04-22 17:07:14,903 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("address")
2025-04-22 17:07:14,903 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-04-22 17:07:14,904 INFO sqlalchemy.engine.Engine 
CREATE TABLE user_account (
	id INTEGER NOT NULL, 
	name VARCHAR(30), 
	fullname VARCHAR, 
	PRIMARY KEY (id)
)


2025-04-22 17:07:14,904 INFO sqlalchemy.engine.Engine [no key 0.00041s] ()
2025-04-22 17:07:14,905 INFO sqlalchemy.engine.Engine 
C

In [34]:
# metadata_obj.drop_all(engine)

## Using ORM Declarative Forms to Define Table Metadata


### Establishing a Declarative Base


In [35]:
from sqlalchemy.orm import DeclarativeBase

In [36]:
class Base(DeclarativeBase):
    pass

In [37]:
Base.metadata

MetaData()

In [38]:
Base.registry

### Declaring Mapped Classes


In [39]:
from typing import Optional, List
from sqlalchemy.orm import Mapped, mapped_column, relationship

In [40]:
class User(Base):
    __tablename__ = "user_account"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(30))
    fullname: Mapped[Optional[str]]

    addresses: Mapped[List["Address"]] = relationship(back_populates="user")

    def __repr__(self) -> str:
        return f"User(id={self.id!r}, name={self.name!r}, fullname={self.fullname!r})"


class Address(Base):
    __tablename__ = "address"

    id: Mapped[int] = mapped_column(primary_key=True)
    email_address: Mapped[str]
    user_id = mapped_column(ForeignKey("user_account.id"))

    user: Mapped[User] = relationship(back_populates="addresses")

    def __repr__(self) -> str:
        return f"Address(id={self.id!r}, email_address={self.email_address!r})"

In [41]:
sandy = User(name="sandy", fullname="Sandy Cheeks")

### Emitting DDL to the database from an ORM mapping


In [42]:
Base.metadata.create_all(engine)

2025-04-22 17:07:14,981 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:14,982 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("user_account")
2025-04-22 17:07:14,983 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-04-22 17:07:14,983 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("address")
2025-04-22 17:07:14,984 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-04-22 17:07:14,984 INFO sqlalchemy.engine.Engine COMMIT


## Table Reflection


In [43]:
some_table = Table("some_table", metadata_obj, autoload_with=engine)

2025-04-22 17:07:14,990 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:14,991 INFO sqlalchemy.engine.Engine PRAGMA main.table_xinfo("some_table")
2025-04-22 17:07:14,991 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-04-22 17:07:14,992 INFO sqlalchemy.engine.Engine SELECT sql FROM  (SELECT * FROM sqlite_master UNION ALL   SELECT * FROM sqlite_temp_master) WHERE name = ? AND type in ('table', 'view')
2025-04-22 17:07:14,992 INFO sqlalchemy.engine.Engine [raw sql] ('some_table',)
2025-04-22 17:07:14,993 INFO sqlalchemy.engine.Engine PRAGMA main.foreign_key_list("some_table")
2025-04-22 17:07:14,993 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-04-22 17:07:14,994 INFO sqlalchemy.engine.Engine PRAGMA temp.foreign_key_list("some_table")
2025-04-22 17:07:14,994 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-04-22 17:07:14,995 INFO sqlalchemy.engine.Engine SELECT sql FROM  (SELECT * FROM sqlite_master UNION ALL   SELECT * FROM sqlite_temp_master) WHERE name = ? AND type i

In [44]:
some_table

Table('some_table', MetaData(), Column('x', INTEGER(), table=<some_table>), Column('y', INTEGER(), table=<some_table>), schema=None)

# Working with Data


## Using INSERT Statements


### The insert() SQL Expression Construct


In [45]:
from sqlalchemy import insert

In [46]:
stmt = insert(user_table).values(name="spongebob", fullname="Spongebob Squarepants")

In [47]:
print(stmt)

INSERT INTO user_account (name, fullname) VALUES (:name, :fullname)


In [48]:
compiled = stmt.compile()

In [49]:
compiled.params

{'name': 'spongebob', 'fullname': 'Spongebob Squarepants'}

### Executing the Statement


In [50]:
with engine.connect() as conn:
    result = conn.execute(stmt)
    conn.commit()

2025-04-22 17:07:15,050 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,051 INFO sqlalchemy.engine.Engine INSERT INTO user_account (name, fullname) VALUES (?, ?)
2025-04-22 17:07:15,051 INFO sqlalchemy.engine.Engine [generated in 0.00165s] ('spongebob', 'Spongebob Squarepants')
2025-04-22 17:07:15,052 INFO sqlalchemy.engine.Engine COMMIT


In [51]:
result.inserted_primary_key

(1,)

### INSERT usually generates the “values” clause automatically


In [52]:
print(insert(user_table))

INSERT INTO user_account (id, name, fullname) VALUES (:id, :name, :fullname)


In [53]:
with engine.connect() as conn:
    result = conn.execute(
        insert(user_table),
        [
            {"name": "sandy", "fullname": "Sandy Cheeks"},
            {"name": "patrick", "fullname": "Patrick Star"},
        ],
    )
    conn.commit()

2025-04-22 17:07:15,073 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,073 INFO sqlalchemy.engine.Engine INSERT INTO user_account (name, fullname) VALUES (?, ?)
2025-04-22 17:07:15,074 INFO sqlalchemy.engine.Engine [generated in 0.00136s] [('sandy', 'Sandy Cheeks'), ('patrick', 'Patrick Star')]
2025-04-22 17:07:15,074 INFO sqlalchemy.engine.Engine COMMIT


In [54]:
from sqlalchemy import select, bindparam

In [55]:
scalar_subq = (
    select(user_table.c.id)
    .where(user_table.c.name == bindparam("username"))
    .scalar_subquery()
)

with engine.connect() as conn:
    result = conn.execute(
        insert(address_table).values(user_id=scalar_subq),
        [
            {
                "username": "spongebob",
                "email_address": "spongebob@sqlalchemy.org",
            },
            {"username": "sandy", "email_address": "sandy@sqlalchemy.org"},
            {"username": "sandy", "email_address": "sandy@squirrelpower.org"},
        ],
    )
    conn.commit()

2025-04-22 17:07:15,090 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,091 INFO sqlalchemy.engine.Engine INSERT INTO address (user_id, email_address) VALUES ((SELECT user_account.id 
FROM user_account 
WHERE user_account.name = ?), ?)
2025-04-22 17:07:15,092 INFO sqlalchemy.engine.Engine [generated in 0.00171s] [('spongebob', 'spongebob@sqlalchemy.org'), ('sandy', 'sandy@sqlalchemy.org'), ('sandy', 'sandy@squirrelpower.org')]
2025-04-22 17:07:15,093 INFO sqlalchemy.engine.Engine COMMIT


In [56]:
print(insert(user_table).values().compile(engine))

INSERT INTO user_account DEFAULT VALUES


### INSERT…RETURNING


In [57]:
insert_stmt = insert(address_table).returning(
    address_table.c.id, address_table.c.email_address
)
print(insert_stmt)

INSERT INTO address (id, user_id, email_address) VALUES (:id, :user_id, :email_address) RETURNING address.id, address.email_address


### INSERT…FROM SELECT


In [58]:
select_stmt = select(user_table.c.id, user_table.c.name + "@alo.com")
insert_stmt = insert(address_table).from_select(
    ["user_id", "email_address"], select_stmt
)
print(insert_stmt)
print(insert_stmt.returning(address_table.c.user_id, address_table.c.email_address))

INSERT INTO address (user_id, email_address) SELECT user_account.id, user_account.name || :name_1 AS anon_1 
FROM user_account
INSERT INTO address (user_id, email_address) SELECT user_account.id, user_account.name || :name_1 AS anon_1 
FROM user_account RETURNING address.user_id, address.email_address


## Using SELECT Statements


### The select() SQL Expression Construct


In [59]:
from sqlalchemy import select

In [60]:
stmt = select(user_table).where(user_table.c.name == "spongebob")
print(stmt)

SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name = :name_1


In [61]:
with engine.connect() as conn:
    result = conn.execute(stmt)

for row in result:
    print(row)

2025-04-22 17:07:15,135 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,136 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name = ?
2025-04-22 17:07:15,136 INFO sqlalchemy.engine.Engine [generated in 0.00140s] ('spongebob',)
2025-04-22 17:07:15,137 INFO sqlalchemy.engine.Engine ROLLBACK
(1, 'spongebob', 'Spongebob Squarepants')


ORM


In [62]:
stmt = select(User).where(User.name == "spongebob")

with Session(engine) as session:
    result = session.execute(stmt)

    for row in result:
        print(row)

2025-04-22 17:07:15,143 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,145 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name = ?
2025-04-22 17:07:15,147 INFO sqlalchemy.engine.Engine [generated in 0.00209s] ('spongebob',)
(User(id=1, name='spongebob', fullname='Spongebob Squarepants'),)
2025-04-22 17:07:15,149 INFO sqlalchemy.engine.Engine ROLLBACK


### Setting the COLUMNS and FROM clause


In [63]:
print(select(user_table))

SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account


In [64]:
print(select(user_table.c.name, user_table.c.fullname))

SELECT user_account.name, user_account.fullname 
FROM user_account


In [65]:
print(select(user_table.c["name", "fullname"]))

SELECT user_account.name, user_account.fullname 
FROM user_account


#### Selecting ORM Entities and Columns


In [66]:
print(select(User))

SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account


In [67]:
with Session(engine) as session:
    row = session.execute(select(User)).first()

row

2025-04-22 17:07:15,195 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,197 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account
2025-04-22 17:07:15,198 INFO sqlalchemy.engine.Engine [generated in 0.00067s] ()
2025-04-22 17:07:15,199 INFO sqlalchemy.engine.Engine ROLLBACK


(User(id=1, name='spongebob', fullname='Spongebob Squarepants'),)

In [68]:
with Session(engine) as session:
    row = session.scalars(select(User)).first()

row

2025-04-22 17:07:15,212 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,213 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account
2025-04-22 17:07:15,214 INFO sqlalchemy.engine.Engine [cached since 0.01657s ago] ()
2025-04-22 17:07:15,216 INFO sqlalchemy.engine.Engine ROLLBACK


User(id=1, name='spongebob', fullname='Spongebob Squarepants')

In [69]:
print(select(User.name, User.fullname))

SELECT user_account.name, user_account.fullname 
FROM user_account


In [70]:
with Session(engine) as session:
    row = session.execute(select(User.name, User.fullname)).first()

row

2025-04-22 17:07:15,235 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,238 INFO sqlalchemy.engine.Engine SELECT user_account.name, user_account.fullname 
FROM user_account
2025-04-22 17:07:15,238 INFO sqlalchemy.engine.Engine [generated in 0.00042s] ()
2025-04-22 17:07:15,239 INFO sqlalchemy.engine.Engine ROLLBACK


('spongebob', 'Spongebob Squarepants')

In [71]:
with Session(engine) as session:
    result = session.execute(
        select(User.name, Address)
        .where(User.id == Address.user_id)
        .order_by(Address.email_address)
    ).all()

    print(result)

2025-04-22 17:07:15,250 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,254 INFO sqlalchemy.engine.Engine SELECT user_account.name, address.id, address.email_address, address.user_id 
FROM user_account, address 
WHERE user_account.id = address.user_id ORDER BY address.email_address
2025-04-22 17:07:15,254 INFO sqlalchemy.engine.Engine [generated in 0.00084s] ()
[('sandy', Address(id=2, email_address='sandy@sqlalchemy.org')), ('sandy', Address(id=3, email_address='sandy@squirrelpower.org')), ('spongebob', Address(id=1, email_address='spongebob@sqlalchemy.org'))]
2025-04-22 17:07:15,256 INFO sqlalchemy.engine.Engine ROLLBACK


#### Selecting from Labeled SQL Expressions


In [72]:
stmt = select(("Username: " + user_table.c.name).label("username")).order_by(
    user_table.c.name
)

with engine.connect() as conn:
    for row in conn.execute(stmt):
        print(row.username)

2025-04-22 17:07:15,267 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,268 INFO sqlalchemy.engine.Engine SELECT ? || user_account.name AS username 
FROM user_account ORDER BY user_account.name
2025-04-22 17:07:15,269 INFO sqlalchemy.engine.Engine [generated in 0.00197s] ('Username: ',)
Username: patrick
Username: sandy
Username: spongebob
2025-04-22 17:07:15,270 INFO sqlalchemy.engine.Engine ROLLBACK


#### Selecting with Textual Column Expressions


In [73]:
from sqlalchemy import text

In [74]:
stmt = select(text("'name'"), user_table.c.name).order_by(user_table.c.name)

with engine.connect() as conn:
    print(conn.execute(stmt).all())

2025-04-22 17:07:15,305 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,307 INFO sqlalchemy.engine.Engine SELECT 'name', user_account.name 
FROM user_account ORDER BY user_account.name
2025-04-22 17:07:15,309 INFO sqlalchemy.engine.Engine [generated in 0.00427s] ()
[('name', 'patrick'), ('name', 'sandy'), ('name', 'spongebob')]
2025-04-22 17:07:15,310 INFO sqlalchemy.engine.Engine ROLLBACK


In [75]:
from sqlalchemy import literal_column

In [76]:
stmt = select(literal_column("'name'").label("n"), user_table.c.name).order_by(
    user_table.c.name
)

with engine.connect() as conn:
    result = conn.execute(stmt).all()

for row in result:
    print(f"{row.n=}, {row.name=}")

2025-04-22 17:07:15,337 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,338 INFO sqlalchemy.engine.Engine SELECT 'name' AS n, user_account.name 
FROM user_account ORDER BY user_account.name
2025-04-22 17:07:15,338 INFO sqlalchemy.engine.Engine [generated in 0.00132s] ()
2025-04-22 17:07:15,339 INFO sqlalchemy.engine.Engine ROLLBACK
row.n='name', row.name='patrick'
row.n='name', row.name='sandy'
row.n='name', row.name='spongebob'


### The WHERE clause


In [77]:
print(user_table.c.name == "squidward")

user_account.name = :name_1


In [78]:
print(address_table.c.id > 10)

address.id > :id_1


In [79]:
print(select(user_table).where(user_table.c.name == "squidward"))

SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name = :name_1


In [80]:
print(
    select(address_table.c.email_address)
    .where(user_table.c.name == "squidward")
    .where(address_table.c.user_id == user_table.c.id)
)

SELECT address.email_address 
FROM address, user_account 
WHERE user_account.name = :name_1 AND address.user_id = user_account.id


In [81]:
print(
    select(address_table.c.email_address).where(
        user_table.c.name == "squidward", address_table.c.user_id == user_table.c.id
    )
)

SELECT address.email_address 
FROM address, user_account 
WHERE user_account.name = :name_1 AND address.user_id = user_account.id


In [82]:
from sqlalchemy import and_, or_

In [83]:
print(
    select(Address.email_address).where(
        and_(
            or_(User.name == "sandey", User.name == "squidward"),
            Address.user_id == User.id,
        )
    )
)

SELECT address.email_address 
FROM address, user_account 
WHERE (user_account.name = :name_1 OR user_account.name = :name_2) AND address.user_id = user_account.id


In [84]:
print(select(User).filter_by(name="spongebob", fullname="Songebob Squarepants"))

SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name = :name_1 AND user_account.fullname = :fullname_1


### Explicit FROM clauses and JOINs


In [85]:
print(select(user_table.c.name))

SELECT user_account.name 
FROM user_account


In [86]:
print(select(user_table.c.name, address_table.c.email_address))

SELECT user_account.name, address.email_address 
FROM user_account, address


Select.join_from() method, which allows us to indicate the left and right side of the JOIN explicitly:


In [87]:
print(
    select(user_table.c.name, address_table.c.email_address).join_from(
        user_table, address_table
    )
)

SELECT user_account.name, address.email_address 
FROM user_account JOIN address ON user_account.id = address.user_id


Select.join() method, which indicates only the right side of the JOIN, the left hand-side is inferred:


In [88]:
print(select(user_table.c.name, address_table.c.email_address).join(address_table))

SELECT user_account.name, address.email_address 
FROM user_account JOIN address ON user_account.id = address.user_id


In [89]:
print(select(address_table.c.email_address).select_from(user_table).join(address_table))

SELECT address.email_address 
FROM user_account JOIN address ON user_account.id = address.user_id


In [90]:
from sqlalchemy import func

In [91]:
print(select(func.count("*")).select_from(user_table))

SELECT count(:count_2) AS count_1 
FROM user_account


#### Setting the ON Clause


Both Select.join() and Select.join_from() accept an additional argument for the ON clause, which is stated using the same SQL Expression mechanics as we saw about in The WHERE clause:


In [92]:
print(
    select(address_table.c.email_address)
    .select_from(user_table)
    .join(address_table, user_table.c.id == address_table.c.user_id)
)

SELECT address.email_address 
FROM user_account JOIN address ON user_account.id = address.user_id


#### OUTER and FULL join


In [93]:
print(select(user_table).join(address_table, isouter=True))

SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account LEFT OUTER JOIN address ON user_account.id = address.user_id


In [94]:
print(select(user_table).join(address_table, full=True))

SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account FULL OUTER JOIN address ON user_account.id = address.user_id


### ORDER BY, GROUP BY, HAVING


#### ORDER BY


In [95]:
print(select(user_table).order_by(user_table.c.name.desc()))

SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account ORDER BY user_account.name DESC


In [96]:
print(select(User).order_by(User.fullname.asc()))

SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account ORDER BY user_account.fullname ASC


#### Aggregate functions with GROUP BY / HAVING


In [97]:
from sqlalchemy import func

count_fn = func.count(user_table.c.id)
print(count_fn)

count(user_account.id)


In [98]:
stmt = (
    select(User.name, func.count(Address.id).label("count"))
    .join(Address)
    .group_by(User.name)
    .having(func.count(Address.id) > 1)
)
with engine.connect() as conn:
    result = conn.execute(stmt)
    print(result.all())

2025-04-22 17:07:15,596 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,601 INFO sqlalchemy.engine.Engine SELECT user_account.name, count(address.id) AS count 
FROM user_account JOIN address ON user_account.id = address.user_id GROUP BY user_account.name 
HAVING count(address.id) > ?
2025-04-22 17:07:15,602 INFO sqlalchemy.engine.Engine [generated in 0.00601s] (1,)
[('sandy', 2)]
2025-04-22 17:07:15,606 INFO sqlalchemy.engine.Engine ROLLBACK


#### Ordering or Grouping by a Label


In [99]:
from sqlalchemy import desc

In [100]:
print(
    select(Address.user_id, func.count(Address.email_address).label("num_addresses"))
    .group_by(Address.user_id)
    .order_by(Address.user_id, desc("num_addresses"))
)

SELECT address.user_id, count(address.email_address) AS num_addresses 
FROM address GROUP BY address.user_id ORDER BY address.user_id, num_addresses DESC


In [101]:
print(
    select(Address.user_id, func.count(Address.email_address).label("num_addresses"))
    .group_by("user_id")
    .order_by("num_addresses")
)

SELECT address.user_id, count(address.email_address) AS num_addresses 
FROM address GROUP BY address.user_id ORDER BY num_addresses


### Using Aliases


In [102]:
user_alias_1 = user_table.alias()
user_alias_2 = user_table.alias()

print(
    select(user_alias_1.c.name, user_alias_2.c.name).join_from(
        user_alias_1, user_alias_2, user_alias_1.c.id > user_alias_2.c.id
    )
)

SELECT user_account_1.name, user_account_2.name AS name_1 
FROM user_account AS user_account_1 JOIN user_account AS user_account_2 ON user_account_1.id > user_account_2.id


#### ORM Entity Aliases


In [103]:
from sqlalchemy.orm import aliased

In [104]:
address_alias_1 = aliased(Address)
address_alias_2 = aliased(Address)
print(
    select(User)
    .join_from(User, address_alias_1)
    .where(address_alias_1.email_address == "patrick@aol.com")
    .join_from(User, address_alias_2)
    .where(address_alias_2.email_address == "patrick@gmail.com")
)

SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account JOIN address AS address_1 ON user_account.id = address_1.user_id JOIN address AS address_2 ON user_account.id = address_2.user_id 
WHERE address_1.email_address = :email_address_1 AND address_2.email_address = :email_address_2


### Subqueries and CTEs


In [105]:
subq = (
    select(func.count(address_table.c.id).label("count"), address_table.c.user_id)
    .group_by(address_table.c.user_id)
    .subquery()
)

In [106]:
print(subq)

SELECT count(address.id) AS count, address.user_id 
FROM address GROUP BY address.user_id


In [107]:
print(select(subq.c.count, subq.c.user_id))

SELECT anon_1.count, anon_1.user_id 
FROM (SELECT count(address.id) AS count, address.user_id AS user_id 
FROM address GROUP BY address.user_id) AS anon_1


In [108]:
stmt = select(user_table.c.name, user_table.c.fullname, subq.c.count).join_from(
    user_table, subq
)

print(stmt)

SELECT user_account.name, user_account.fullname, anon_1.count 
FROM user_account JOIN (SELECT count(address.id) AS count, address.user_id AS user_id 
FROM address GROUP BY address.user_id) AS anon_1 ON user_account.id = anon_1.user_id


#### Common Table Expressions (CTEs)


In [109]:
subq = (
    select(func.count(address_table.c.id).label("count"), address_table.c.user_id)
    .group_by(address_table.c.user_id)
    .cte()
)

stmt = select(user_table.c.name, user_table.c.fullname, subq.c.count).join_from(
    user_table, subq
)

print(stmt)

WITH anon_1 AS 
(SELECT count(address.id) AS count, address.user_id AS user_id 
FROM address GROUP BY address.user_id)
 SELECT user_account.name, user_account.fullname, anon_1.count 
FROM user_account JOIN anon_1 ON user_account.id = anon_1.user_id


#### ORM Entity Subqueries/CTEs


In [110]:
subq = select(Address).where(~Address.email_address.like("%@aol.com")).subquery()
address_subq = aliased(Address, subq)
stmt = (
    select(User, address_subq)
    .join_from(User, address_subq)
    .order_by(User.id, address_subq.id)
)
with Session(engine) as session:
    for user, address in session.execute(stmt):
        print(f"{user} {address}")

2025-04-22 17:07:15,754 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,759 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname, anon_1.id AS id_1, anon_1.email_address, anon_1.user_id 
FROM user_account JOIN (SELECT address.id AS id, address.email_address AS email_address, address.user_id AS user_id 
FROM address 
WHERE address.email_address NOT LIKE ?) AS anon_1 ON user_account.id = anon_1.user_id ORDER BY user_account.id, anon_1.id
2025-04-22 17:07:15,760 INFO sqlalchemy.engine.Engine [generated in 0.00137s] ('%@aol.com',)
User(id=1, name='spongebob', fullname='Spongebob Squarepants') Address(id=1, email_address='spongebob@sqlalchemy.org')
User(id=2, name='sandy', fullname='Sandy Cheeks') Address(id=2, email_address='sandy@sqlalchemy.org')
User(id=2, name='sandy', fullname='Sandy Cheeks') Address(id=3, email_address='sandy@squirrelpower.org')
2025-04-22 17:07:15,762 INFO sqlalchemy.engine.Engine ROLLBACK


In [111]:
cte_obj = select(Address).where(~Address.email_address.like("%@aol.com")).cte()
address_cte = aliased(Address, cte_obj)

stmt = (
    select(User, address_cte)
    .join_from(User, address_cte)
    .order_by(User.id, address_cte.id)
)
with Session(engine) as session:
    for user, address in session.execute(stmt):
        print(f"{user} {address}")

2025-04-22 17:07:15,773 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,776 INFO sqlalchemy.engine.Engine WITH anon_1 AS 
(SELECT address.id AS id, address.email_address AS email_address, address.user_id AS user_id 
FROM address 
WHERE address.email_address NOT LIKE ?)
 SELECT user_account.id, user_account.name, user_account.fullname, anon_1.id AS id_1, anon_1.email_address, anon_1.user_id 
FROM user_account JOIN anon_1 ON user_account.id = anon_1.user_id ORDER BY user_account.id, anon_1.id
2025-04-22 17:07:15,777 INFO sqlalchemy.engine.Engine [generated in 0.00134s] ('%@aol.com',)
User(id=1, name='spongebob', fullname='Spongebob Squarepants') Address(id=1, email_address='spongebob@sqlalchemy.org')
User(id=2, name='sandy', fullname='Sandy Cheeks') Address(id=2, email_address='sandy@sqlalchemy.org')
User(id=2, name='sandy', fullname='Sandy Cheeks') Address(id=3, email_address='sandy@squirrelpower.org')
2025-04-22 17:07:15,781 INFO sqlalchemy.engine.Engine ROLLBACK


### Scalar and Correlated Subqueries


In [112]:
subq = (
    select(func.count(address_table.c.id))
    .where(user_table.c.id == address_table.c.user_id)
    .scalar_subquery()
)

In [113]:
print(subq)

(SELECT count(address.id) AS count_1 
FROM address, user_account 
WHERE user_account.id = address.user_id)


In [114]:
print(subq == 5)

(SELECT count(address.id) AS count_1 
FROM address, user_account 
WHERE user_account.id = address.user_id) = :param_1


In [115]:
stmt = select(user_table.c.name, subq.label("address_count"))

In [116]:
print(stmt)

SELECT user_account.name, (SELECT count(address.id) AS count_1 
FROM address 
WHERE user_account.id = address.user_id) AS address_count 
FROM user_account


In [117]:
# stmt = (
#     select(
#         user_table.c.name,
#         address_table.c.email_address,
#         subq.label("address_count"),
#     )
#     .join_from(user_table, address_table)
#     .order_by(user_table.c.id, address_table.c.id)
# )
# print(stmt)

In [118]:
subq = (
    select(func.count(address_table.c.id))
    .where(address_table.c.user_id == user_table.c.id)
    .scalar_subquery()
    .correlate(user_table)
)

In [119]:
print(subq)

(SELECT count(address.id) AS count_1 
FROM address, user_account 
WHERE address.user_id = user_account.id)


In [120]:
with engine.connect() as conn:
    result = conn.execute(
        select(
            user_table.c.name,
            address_table.c.email_address,
            subq.label("address_count"),
        )
        .join_from(user_table, address_table)
        .order_by(user_table.c.id, address_table.c.id)
    )
    print(result.all())

2025-04-22 17:07:15,865 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,866 INFO sqlalchemy.engine.Engine SELECT user_account.name, address.email_address, (SELECT count(address.id) AS count_1 
FROM address 
WHERE address.user_id = user_account.id) AS address_count 
FROM user_account JOIN address ON user_account.id = address.user_id ORDER BY user_account.id, address.id
2025-04-22 17:07:15,867 INFO sqlalchemy.engine.Engine [generated in 0.00143s] ()
[('spongebob', 'spongebob@sqlalchemy.org', 1), ('sandy', 'sandy@sqlalchemy.org', 2), ('sandy', 'sandy@squirrelpower.org', 2)]
2025-04-22 17:07:15,868 INFO sqlalchemy.engine.Engine ROLLBACK


#### LATERAL correlation


In [121]:
subq = (
    select(
        func.count(address_table.c.id).label("address_count"),
        address_table.c.email_address,
        address_table.c.user_id,
    )
    .where(user_table.c.id == address_table.c.user_id)
    .lateral()
)
stmt = (
    select(user_table.c.name, subq.c.address_count, subq.c.email_address)
    .join_from(user_table, subq)
    .order_by(user_table.c.id, subq.c.email_address)
)
print(stmt)

SELECT user_account.name, anon_1.address_count, anon_1.email_address 
FROM user_account JOIN LATERAL (SELECT count(address.id) AS address_count, address.email_address AS email_address, address.user_id AS user_id 
FROM address 
WHERE user_account.id = address.user_id) AS anon_1 ON user_account.id = anon_1.user_id ORDER BY user_account.id, anon_1.email_address


In [122]:
from sqlalchemy import union_all

In [123]:
stmt1 = select(user_table).where(user_table.c.name == "sandy")
stmt2 = select(user_table).where(user_table.c.name == "spongebob")
u = union_all(stmt1, stmt2)
with engine.connect() as conn:
    result = conn.execute(u)
    print(result.all())

2025-04-22 17:07:15,905 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,906 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name = ? UNION ALL SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name = ?
2025-04-22 17:07:15,906 INFO sqlalchemy.engine.Engine [generated in 0.00219s] ('sandy', 'spongebob')
[(2, 'sandy', 'Sandy Cheeks'), (1, 'spongebob', 'Spongebob Squarepants')]
2025-04-22 17:07:15,907 INFO sqlalchemy.engine.Engine ROLLBACK


In [124]:
u_subq = u.subquery()
stmt = (
    select(u_subq.c.name, address_table.c.email_address)
    .join_from(address_table, u_subq)
    .order_by(u_subq.c.name, address_table.c.email_address)
)
with engine.connect() as conn:
    result = conn.execute(stmt)
    print(result.all())

2025-04-22 17:07:15,919 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,920 INFO sqlalchemy.engine.Engine SELECT anon_1.name, address.email_address 
FROM address JOIN (SELECT user_account.id AS id, user_account.name AS name, user_account.fullname AS fullname 
FROM user_account 
WHERE user_account.name = ? UNION ALL SELECT user_account.id AS id, user_account.name AS name, user_account.fullname AS fullname 
FROM user_account 
WHERE user_account.name = ?) AS anon_1 ON anon_1.id = address.user_id ORDER BY anon_1.name, address.email_address
2025-04-22 17:07:15,921 INFO sqlalchemy.engine.Engine [generated in 0.00181s] ('sandy', 'spongebob')
[('sandy', 'sandy@sqlalchemy.org'), ('sandy', 'sandy@squirrelpower.org'), ('spongebob', 'spongebob@sqlalchemy.org')]
2025-04-22 17:07:15,922 INFO sqlalchemy.engine.Engine ROLLBACK


#### Selecting ORM Entities from Unions


In [125]:
stmt1 = select(User).where(User.name == "sandy")
stmt2 = select(User).where(User.name == "spongebob")
u = union_all(stmt1, stmt2)

In [126]:
orm_stmt = select(User).from_statement(u)
with Session(engine) as session:
    for obj in session.execute(orm_stmt).scalars():
        print(obj)

2025-04-22 17:07:15,942 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,945 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name = ? UNION ALL SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name = ?
2025-04-22 17:07:15,946 INFO sqlalchemy.engine.Engine [generated in 0.00165s] ('sandy', 'spongebob')
User(id=2, name='sandy', fullname='Sandy Cheeks')
User(id=1, name='spongebob', fullname='Spongebob Squarepants')
2025-04-22 17:07:15,948 INFO sqlalchemy.engine.Engine ROLLBACK


In [127]:
user_alias = aliased(User, u.subquery())
orm_stmt = select(user_alias).order_by(user_alias.id)
with Session(engine) as session:
    for obj in session.execute(orm_stmt).scalars():
        print(obj)

2025-04-22 17:07:15,959 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,963 INFO sqlalchemy.engine.Engine SELECT anon_1.id, anon_1.name, anon_1.fullname 
FROM (SELECT user_account.id AS id, user_account.name AS name, user_account.fullname AS fullname 
FROM user_account 
WHERE user_account.name = ? UNION ALL SELECT user_account.id AS id, user_account.name AS name, user_account.fullname AS fullname 
FROM user_account 
WHERE user_account.name = ?) AS anon_1 ORDER BY anon_1.id
2025-04-22 17:07:15,964 INFO sqlalchemy.engine.Engine [generated in 0.00075s] ('sandy', 'spongebob')
User(id=1, name='spongebob', fullname='Spongebob Squarepants')
User(id=2, name='sandy', fullname='Sandy Cheeks')
2025-04-22 17:07:15,966 INFO sqlalchemy.engine.Engine ROLLBACK


### EXISTS subqueries


In [128]:
subq = (
    select(func.count(address_table.c.id))
    .where(user_table.c.id == address_table.c.user_id)
    .group_by(address_table.c.user_id)
    .having(func.count(address_table.c.id) > 1)
).exists()
with engine.connect() as conn:
    result = conn.execute(select(user_table.c.name).where(subq))
    print(result.all())

2025-04-22 17:07:15,977 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,978 INFO sqlalchemy.engine.Engine SELECT user_account.name 
FROM user_account 
WHERE EXISTS (SELECT count(address.id) AS count_1 
FROM address 
WHERE user_account.id = address.user_id GROUP BY address.user_id 
HAVING count(address.id) > ?)
2025-04-22 17:07:15,979 INFO sqlalchemy.engine.Engine [generated in 0.00190s] (1,)
[('sandy',)]
2025-04-22 17:07:15,981 INFO sqlalchemy.engine.Engine ROLLBACK


In [129]:
subq = (
    select(address_table.c.id).where(user_table.c.id == address_table.c.user_id)
).exists()
with engine.connect() as conn:
    result = conn.execute(select(user_table.c.name).where(~subq))
    print(result.all())

2025-04-22 17:07:15,992 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:15,994 INFO sqlalchemy.engine.Engine SELECT user_account.name 
FROM user_account 
WHERE NOT (EXISTS (SELECT address.id 
FROM address 
WHERE user_account.id = address.user_id))
2025-04-22 17:07:15,995 INFO sqlalchemy.engine.Engine [generated in 0.00277s] ()
[('patrick',)]
2025-04-22 17:07:15,996 INFO sqlalchemy.engine.Engine ROLLBACK


### Working with SQL Functions


In [130]:
print(select(func.count()).select_from(user_table))

SELECT count(*) AS count_1 
FROM user_account


In [131]:
print(select(func.lower("A String With Much UPPERCASE")))

SELECT lower(:lower_2) AS lower_1


In [132]:
stmt = select(func.now())

In [133]:
with engine.connect() as conn:
    result = conn.execute(stmt)
    print(result.all())

2025-04-22 17:07:16,034 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:16,037 INFO sqlalchemy.engine.Engine SELECT CURRENT_TIMESTAMP AS now_1
2025-04-22 17:07:16,038 INFO sqlalchemy.engine.Engine [generated in 0.00381s] ()
[(datetime.datetime(2025, 4, 22, 14, 7, 16),)]
2025-04-22 17:07:16,038 INFO sqlalchemy.engine.Engine ROLLBACK


#### Functions Have Return Types


In [134]:
func.now().type

DateTime()

In [135]:
func.run_some_calculation().type

NullType()

In [136]:
from sqlalchemy import JSON

function_expr = func.json_object('{a, 1, b, "def", c, 3.5}', type_=JSON)

In [137]:
stmt = select(function_expr["def"])
print(stmt)

SELECT json_object(:json_object_1)[:json_object_2] AS anon_1


#### Built-in Functions Have Pre-Configured Return Types


In [138]:
m1 = func.max(Column("some_int", Integer))
m1.type

Integer()

In [139]:
m2 = func.max(Column("some_str", String))
m2.type

String()

In [140]:
func.now().type

DateTime()

In [141]:
func.current_date().type

Date()

In [142]:
func.concat("x", "y").type

String()

In [143]:
func.upper("lowercase").type

NullType()

In [144]:
print(select(func.upper("lowercase") + " suffix"))

SELECT upper(:upper_1) || :upper_2 AS anon_1


#### Advanced SQL Function Techniques


##### Using Window Functions


In [145]:
stmt = (
    select(
        func.row_number().over(partition_by=user_table.c.name),
        user_table.c.name,
        address_table.c.email_address,
    )
    .select_from(user_table)
    .join(address_table)
)
with engine.connect() as conn:
    result = conn.execute(stmt)
    print(result.all())

2025-04-22 17:07:16,165 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:16,167 INFO sqlalchemy.engine.Engine SELECT row_number() OVER (PARTITION BY user_account.name) AS anon_1, user_account.name, address.email_address 
FROM user_account JOIN address ON user_account.id = address.user_id
2025-04-22 17:07:16,168 INFO sqlalchemy.engine.Engine [generated in 0.00327s] ()
[(1, 'sandy', 'sandy@sqlalchemy.org'), (2, 'sandy', 'sandy@squirrelpower.org'), (1, 'spongebob', 'spongebob@sqlalchemy.org')]
2025-04-22 17:07:16,169 INFO sqlalchemy.engine.Engine ROLLBACK


In [146]:
stmt = (
    select(
        func.count().over(order_by=user_table.c.name),
        user_table.c.name,
        address_table.c.email_address,
    )
    .select_from(user_table)
    .join(address_table)
)
with engine.connect() as conn:
    result = conn.execute(stmt)
    print(result.all())

2025-04-22 17:07:16,179 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:16,181 INFO sqlalchemy.engine.Engine SELECT count(*) OVER (ORDER BY user_account.name) AS anon_1, user_account.name, address.email_address 
FROM user_account JOIN address ON user_account.id = address.user_id
2025-04-22 17:07:16,182 INFO sqlalchemy.engine.Engine [generated in 0.00321s] ()
[(2, 'sandy', 'sandy@sqlalchemy.org'), (2, 'sandy', 'sandy@squirrelpower.org'), (3, 'spongebob', 'spongebob@sqlalchemy.org')]
2025-04-22 17:07:16,183 INFO sqlalchemy.engine.Engine ROLLBACK


##### Special Modifiers WITHIN GROUP, FILTER


In [147]:
print(
    func.unnest(
        func.percentile_disc([0.25, 0.5, 0.75, 1]).within_group(user_table.c.name)
    )
)

unnest(percentile_disc(:percentile_disc_1) WITHIN GROUP (ORDER BY user_account.name))


In [148]:
stmt = (
    select(
        func.count(address_table.c.email_address).filter(user_table.c.name == "sandy"),
        func.count(address_table.c.email_address).filter(
            user_table.c.name == "spongebob"
        ),
    )
    .select_from(user_table)
    .join(address_table)
)

with engine.connect() as conn:
    result = conn.execute(stmt)
    print(result.all())

2025-04-22 17:07:16,203 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:16,204 INFO sqlalchemy.engine.Engine SELECT count(address.email_address) FILTER (WHERE user_account.name = ?) AS anon_1, count(address.email_address) FILTER (WHERE user_account.name = ?) AS anon_2 
FROM user_account JOIN address ON user_account.id = address.user_id
2025-04-22 17:07:16,204 INFO sqlalchemy.engine.Engine [generated in 0.00133s] ('sandy', 'spongebob')
[(2, 1)]
2025-04-22 17:07:16,205 INFO sqlalchemy.engine.Engine ROLLBACK


##### Table-Valued Functions


In [149]:
onetwothree = func.json_each('["one", "two", "three"]').table_valued("value")
stmt = select(onetwothree).where(onetwothree.c.value.in_(["two", "three"]))
with engine.connect() as conn:
    result = conn.execute(stmt)
    print(result.all())

2025-04-22 17:07:16,217 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:16,218 INFO sqlalchemy.engine.Engine SELECT anon_1.value 
FROM json_each(?) AS anon_1 
WHERE anon_1.value IN (?, ?)
2025-04-22 17:07:16,219 INFO sqlalchemy.engine.Engine [generated in 0.00199s] ('["one", "two", "three"]', 'two', 'three')
[('two',), ('three',)]
2025-04-22 17:07:16,219 INFO sqlalchemy.engine.Engine ROLLBACK


##### Column Valued Functions - Table Valued Function as a Scalar Column


In [150]:
from sqlalchemy import select, func

stmt = select(func.json_array_elements('["one", "two"]').column_valued("x"))
print(stmt)

SELECT x 
FROM json_array_elements(:json_array_elements_1) AS x


In [151]:
from sqlalchemy.dialects import oracle

stmt = select(func.scalar_strings(5).column_valued("s"))
print(stmt.compile(dialect=oracle.dialect()))

SELECT s.COLUMN_VALUE 
FROM TABLE (scalar_strings(:scalar_strings_1)) s


### Data Casts and Type Coercion


In [152]:
from sqlalchemy import cast

In [153]:
stmt = select(cast(user_table.c.id, String))

with engine.connect() as conn:
    result = conn.execute(stmt)
    print(result.all())

2025-04-22 17:07:16,294 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:16,295 INFO sqlalchemy.engine.Engine SELECT CAST(user_account.id AS VARCHAR) AS id 
FROM user_account
2025-04-22 17:07:16,295 INFO sqlalchemy.engine.Engine [generated in 0.00132s] ()
[('1',), ('2',), ('3',)]
2025-04-22 17:07:16,296 INFO sqlalchemy.engine.Engine ROLLBACK


In [154]:
from sqlalchemy import JSON

In [155]:
print(cast("{'a':'b'}", JSON)["a"])

CAST(:param_1 AS JSON)[:param_2]


#### type_coerce() - a Python-only “cast”


In [156]:
import json
from sqlalchemy import JSON
from sqlalchemy import type_coerce
from sqlalchemy.dialects import mysql

s = select(type_coerce({"some_key": {"foo": "bar"}}, JSON)["some_key"])
print(s.compile(dialect=mysql.dialect()))

SELECT JSON_EXTRACT(%s, %s) AS anon_1


## Using UPDATE and DELETE Statements


### The update() SQL Expression Construct


In [157]:
from sqlalchemy import update

In [158]:
stmt = (
    update(user_table)
    .where(user_table.c.name == "patrick")
    .values(fullname="Pastrick the Star")
)

print(stmt)

UPDATE user_account SET fullname=:fullname WHERE user_account.name = :name_1


In [159]:
stmt = update(user_table).values(fullname="Username: " + user_table.c.name)

print(stmt)

UPDATE user_account SET fullname=(:name_1 || user_account.name)


In [160]:
from sqlalchemy import bindparam

In [161]:
stmt = (
    update(user_table)
    .where(user_table.c.name == bindparam("oldname"))
    .values(name=bindparam("newname"))
)

with engine.connect() as conn:
    conn.execute(
        stmt,
        [
            {"oldname": "jack", "newname": "ed"},
            {"oldname": "wendy", "newname": "mary"},
            {"oldname": "jim", "newname": "jake"},
        ],
    )

2025-04-22 17:07:16,406 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:16,407 INFO sqlalchemy.engine.Engine UPDATE user_account SET name=? WHERE user_account.name = ?
2025-04-22 17:07:16,407 INFO sqlalchemy.engine.Engine [generated in 0.00164s] [('ed', 'jack'), ('mary', 'wendy'), ('jake', 'jim')]
2025-04-22 17:07:16,408 INFO sqlalchemy.engine.Engine ROLLBACK


#### Correlated Updates


In [162]:
scalar_subq = (
    select(address_table.c.email_address)
    .where(address_table.c.user_id == user_table.c.id)
    .order_by(address_table.c.id)
    .limit(1)
    .scalar_subquery()
)

update_stmt = update(user_table).values(fullname=scalar_subq)

print(update_stmt)

UPDATE user_account SET fullname=(SELECT address.email_address 
FROM address 
WHERE address.user_id = user_account.id ORDER BY address.id
 LIMIT :param_1)


#### UPDATE..FROM


In [163]:
update_stmt = (
    update(user_table)
    .where(user_table.c.id == address_table.c.id)
    .where(address_table.c.email_address == "patrick@aol.com")
    .values(fullname="Pat")
)

print(update_stmt)

UPDATE user_account SET fullname=:fullname FROM address WHERE user_account.id = address.id AND address.email_address = :email_address_1


In [164]:
update_stmt = (
    update(user_table)
    .where(user_table.c.id == address_table.c.id)
    .where(address_table.c.email_address == "patrick@aol.com")
    .values({user_table.c.name: "Pat", address_table.c.email_address: "pat@aol.com"})
)

In [165]:
from sqlalchemy.dialects import mysql

print(update_stmt.compile(dialect=mysql.dialect()))

UPDATE user_account, address SET address.email_address=%s, user_account.name=%s WHERE user_account.id = address.id AND address.email_address = %s


#### Parameter Ordered Updates


In [166]:
update_stmt = update(some_table).ordered_values(
    (some_table.c.y, 20), (some_table.c.x, some_table.c.y + 10)
)
print(update_stmt)

UPDATE some_table SET y=:y, x=(some_table.y + :y_1)


### The delete() SQL Expression Construct


In [167]:
from sqlalchemy import delete

In [168]:
stmt = delete(user_table).where(user_table.c.name == "patrick")

print(stmt)

DELETE FROM user_account WHERE user_account.name = :name_1


#### Multiple Table Deletes


In [169]:
delete_stmt = (
    delete(user_table)
    .where(user_table.c.id == address_table.c.user_id)
    .where(address_table.c.email_address == "patrick@aol.com")
)
from sqlalchemy.dialects import mysql

print(delete_stmt.compile(dialect=mysql.dialect()))

DELETE FROM user_account USING user_account, address WHERE user_account.id = address.user_id AND address.email_address = %s


### Getting Affected Row Count from UPDATE, DELETE


In [170]:
with engine.begin() as conn:
    result = conn.execute(
        update(user_table)
        .values(fullname="Patrick McStar")
        .where(user_table.c.name == "patrick")
    )
    print(result.rowcount)

2025-04-22 17:07:16,508 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:16,510 INFO sqlalchemy.engine.Engine UPDATE user_account SET fullname=? WHERE user_account.name = ?
2025-04-22 17:07:16,511 INFO sqlalchemy.engine.Engine [generated in 0.00106s] ('Patrick McStar', 'patrick')
1
2025-04-22 17:07:16,511 INFO sqlalchemy.engine.Engine COMMIT


### Using RETURNING with UPDATE, DELETE


In [171]:
update_stmt = (
    update(user_table)
    .where(user_table.c.name == "patrick")
    .values(fullname="Patrick the Star")
    .returning(user_table.c.id, user_table.c.name)
)
print(update_stmt)

UPDATE user_account SET fullname=:fullname WHERE user_account.name = :name_1 RETURNING user_account.id, user_account.name


In [172]:
delete_stmt = (
    delete(user_table)
    .where(user_table.c.name == "patrick")
    .returning(user_table.c.id, user_table.c.name)
)
print(delete_stmt)

DELETE FROM user_account WHERE user_account.name = :name_1 RETURNING user_account.id, user_account.name


# Data Manipulation with the ORM


## Inserting Rows using the ORM Unit of Work pattern


### Instances of Classes represent Rows


In [173]:
squidward = User(name="squidward", fullname="Squidward Tentacles")

krabs = User(name="ehkrabs", fullname="Eugene H. Krabs")

In [174]:
squidward

User(id=None, name='squidward', fullname='Squidward Tentacles')

### Adding objects to a Session


In [175]:
session = Session(engine)

In [176]:
# def get_session(engine):
#     return Session(engine)

In [177]:
# with get_session(engine) as session:
#     session.add(krabs)
#     session.commit()

In [178]:
session.add(squidward)
session.add(krabs)

In [179]:
session.new

IdentitySet([User(id=None, name='squidward', fullname='Squidward Tentacles'), User(id=None, name='ehkrabs', fullname='Eugene H. Krabs')])

### Flushing


In [180]:
session.flush()

2025-04-22 17:07:16,606 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:16,608 INFO sqlalchemy.engine.Engine INSERT INTO user_account (name, fullname) VALUES (?, ?) RETURNING id
2025-04-22 17:07:16,609 INFO sqlalchemy.engine.Engine [generated in 0.00010s (insertmanyvalues) 1/2 (ordered; batch not supported)] ('squidward', 'Squidward Tentacles')
2025-04-22 17:07:16,609 INFO sqlalchemy.engine.Engine INSERT INTO user_account (name, fullname) VALUES (?, ?) RETURNING id
2025-04-22 17:07:16,610 INFO sqlalchemy.engine.Engine [insertmanyvalues 2/2 (ordered; batch not supported)] ('ehkrabs', 'Eugene H. Krabs')


### Autogenerated primary key attributes


In [181]:
squidward.id

4

In [182]:
krabs.id

5

#### Getting Objects by Primary Key from the Identity Map


In [183]:
some_squidward = session.get(User, 4)

some_squidward

User(id=4, name='squidward', fullname='Squidward Tentacles')

In [184]:
some_squidward is squidward

True

#### Committing


In [185]:
session.commit()

2025-04-22 17:07:16,676 INFO sqlalchemy.engine.Engine COMMIT


## Updating ORM Objects using the Unit of Work pattern


In [186]:
sandy = session.execute(select(User).filter_by(name="sandy")).scalar_one()

2025-04-22 17:07:16,686 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:16,691 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name = ?
2025-04-22 17:07:16,694 INFO sqlalchemy.engine.Engine [cached since 1.549s ago] ('sandy',)


In [187]:
sandy

User(id=2, name='sandy', fullname='Sandy Cheeks')

In [188]:
sandy.fullname = "Sandy Squirrel"

In [189]:
sandy in session.dirty

True

In [190]:
sandy_fullname = session.execute(select(User.fullname).where(User.id == 2)).scalar_one()

2025-04-22 17:07:16,733 INFO sqlalchemy.engine.Engine UPDATE user_account SET fullname=? WHERE user_account.id = ?
2025-04-22 17:07:16,734 INFO sqlalchemy.engine.Engine [generated in 0.00095s] ('Sandy Squirrel', 2)
2025-04-22 17:07:16,735 INFO sqlalchemy.engine.Engine SELECT user_account.fullname 
FROM user_account 
WHERE user_account.id = ?
2025-04-22 17:07:16,735 INFO sqlalchemy.engine.Engine [generated in 0.00046s] (2,)


In [191]:
print(sandy_fullname)

Sandy Squirrel


In [192]:
sandy in session.dirty

False

## Deleting ORM Objects using the Unit of Work pattern


In [193]:
patrick = session.get(User, 3)

2025-04-22 17:07:16,759 INFO sqlalchemy.engine.Engine SELECT user_account.id AS user_account_id, user_account.name AS user_account_name, user_account.fullname AS user_account_fullname 
FROM user_account 
WHERE user_account.id = ?
2025-04-22 17:07:16,760 INFO sqlalchemy.engine.Engine [generated in 0.00113s] (3,)


In [194]:
session.delete(patrick)

In [195]:
session.execute(select(User).where(User.name == "patrick")).first()

2025-04-22 17:07:16,781 INFO sqlalchemy.engine.Engine SELECT address.id AS address_id, address.email_address AS address_email_address, address.user_id AS address_user_id 
FROM address 
WHERE ? = address.user_id
2025-04-22 17:07:16,782 INFO sqlalchemy.engine.Engine [generated in 0.00101s] (3,)
2025-04-22 17:07:16,783 INFO sqlalchemy.engine.Engine DELETE FROM user_account WHERE user_account.id = ?
2025-04-22 17:07:16,784 INFO sqlalchemy.engine.Engine [generated in 0.00053s] (3,)
2025-04-22 17:07:16,784 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name = ?
2025-04-22 17:07:16,785 INFO sqlalchemy.engine.Engine [cached since 1.64s ago] ('patrick',)


In [196]:
patrick in session

False

## Bulk / Multi Row INSERT, upsert, UPDATE and DELETE


## Rolling Back


In [197]:
session.rollback()

2025-04-22 17:07:16,801 INFO sqlalchemy.engine.Engine ROLLBACK


In [198]:
sandy.__dict__

{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState at 0x7f9fe7fead50>}

In [199]:
sandy.fullname

2025-04-22 17:07:16,823 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:16,824 INFO sqlalchemy.engine.Engine SELECT user_account.id AS user_account_id, user_account.name AS user_account_name, user_account.fullname AS user_account_fullname 
FROM user_account 
WHERE user_account.id = ?
2025-04-22 17:07:16,824 INFO sqlalchemy.engine.Engine [generated in 0.00050s] (2,)


'Sandy Cheeks'

In [200]:
sandy.__dict__

{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState at 0x7f9fe7fead50>,
 'name': 'sandy',
 'fullname': 'Sandy Cheeks',
 'id': 2}

In [201]:
patrick in session

True

In [202]:
session.execute(select(User).where(User.name == "patrick")).scalar_one() is patrick

2025-04-22 17:07:16,848 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name = ?
2025-04-22 17:07:16,849 INFO sqlalchemy.engine.Engine [cached since 1.704s ago] ('patrick',)


True

## Closing a Session


In [203]:
session.close()

2025-04-22 17:07:16,857 INFO sqlalchemy.engine.Engine ROLLBACK


In [204]:
# squidward.name

In [205]:
session.add(squidward)
squidward.name

2025-04-22 17:07:16,877 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:16,878 INFO sqlalchemy.engine.Engine SELECT user_account.id AS user_account_id, user_account.name AS user_account_name, user_account.fullname AS user_account_fullname 
FROM user_account 
WHERE user_account.id = ?
2025-04-22 17:07:16,878 INFO sqlalchemy.engine.Engine [cached since 0.05465s ago] (4,)


'squidward'

# Working with ORM Related Objects


## Persisting and Loading Relationships


In [206]:
u1 = User(name="pkrabs", fullname="Pearl Krabs")

In [207]:
u1.addresses

[]

In [208]:
a1 = Address(email_address="peral.krabs@gmail.com")

In [209]:
u1.addresses.append(a1)

In [210]:
u1.addresses

[Address(id=None, email_address='peral.krabs@gmail.com')]

In [211]:
a1.user

User(id=None, name='pkrabs', fullname='Pearl Krabs')

In [212]:
a2 = Address(email_address="peral@aol.com", user=u1)

In [213]:
u1.addresses

[Address(id=None, email_address='peral.krabs@gmail.com'),
 Address(id=None, email_address='peral@aol.com')]

### Cascading Objects into the Session


In [214]:
session.add(u1)

In [215]:
u1 in session

True

In [216]:
a1 in session

True

In [217]:
a2 in session

True

In [218]:
print(u1.id)
print(a1.user_id)

None
None


In [219]:
session.commit()

2025-04-22 17:07:16,998 INFO sqlalchemy.engine.Engine INSERT INTO user_account (name, fullname) VALUES (?, ?)
2025-04-22 17:07:16,999 INFO sqlalchemy.engine.Engine [generated in 0.00087s] ('pkrabs', 'Pearl Krabs')
2025-04-22 17:07:17,001 INFO sqlalchemy.engine.Engine INSERT INTO address (email_address, user_id) VALUES (?, ?) RETURNING id
2025-04-22 17:07:17,002 INFO sqlalchemy.engine.Engine [generated in 0.00011s (insertmanyvalues) 1/2 (ordered; batch not supported)] ('peral.krabs@gmail.com', 6)
2025-04-22 17:07:17,002 INFO sqlalchemy.engine.Engine INSERT INTO address (email_address, user_id) VALUES (?, ?) RETURNING id
2025-04-22 17:07:17,002 INFO sqlalchemy.engine.Engine [insertmanyvalues 2/2 (ordered; batch not supported)] ('peral@aol.com', 6)
2025-04-22 17:07:17,003 INFO sqlalchemy.engine.Engine COMMIT


## Loading Relationships


In [220]:
u1.id

2025-04-22 17:07:17,011 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-22 17:07:17,012 INFO sqlalchemy.engine.Engine SELECT user_account.id AS user_account_id, user_account.name AS user_account_name, user_account.fullname AS user_account_fullname 
FROM user_account 
WHERE user_account.id = ?
2025-04-22 17:07:17,013 INFO sqlalchemy.engine.Engine [cached since 0.1889s ago] (6,)


6

In [221]:
u1.addresses

2025-04-22 17:07:17,025 INFO sqlalchemy.engine.Engine SELECT address.id AS address_id, address.email_address AS address_email_address, address.user_id AS address_user_id 
FROM address 
WHERE ? = address.user_id
2025-04-22 17:07:17,025 INFO sqlalchemy.engine.Engine [cached since 0.2449s ago] (6,)


[Address(id=4, email_address='peral.krabs@gmail.com'),
 Address(id=5, email_address='peral@aol.com')]

In [222]:
u1.addresses

[Address(id=4, email_address='peral.krabs@gmail.com'),
 Address(id=5, email_address='peral@aol.com')]

In [223]:
a1

Address(id=4, email_address='peral.krabs@gmail.com')

In [224]:
a2

Address(id=5, email_address='peral@aol.com')

## Using Relationships in Queries


### Using Relationships to Join


In [225]:
print(select(Address.email_address).select_from(User).join(User.addresses))

SELECT address.email_address 
FROM user_account JOIN address ON user_account.id = address.user_id


In [226]:
print(select(Address.email_address).join_from(User, Address))

SELECT address.email_address 
FROM user_account JOIN address ON user_account.id = address.user_id


### Relationship WHERE Operators


In [227]:
stmt = select(User.fullname).where(
    User.addresses.any(Address.email_address == "squirrel@squirrelpower.org")
)
session.execute(stmt).all()

2025-04-22 17:07:17,086 INFO sqlalchemy.engine.Engine SELECT user_account.fullname 
FROM user_account 
WHERE EXISTS (SELECT 1 
FROM address 
WHERE user_account.id = address.user_id AND address.email_address = ?)
2025-04-22 17:07:17,087 INFO sqlalchemy.engine.Engine [generated in 0.00130s] ('squirrel@squirrelpower.org',)


[]

In [228]:
stmt = select(User.fullname).where(~User.addresses.any())
session.execute(stmt).all()

2025-04-22 17:07:17,099 INFO sqlalchemy.engine.Engine SELECT user_account.fullname 
FROM user_account 
WHERE NOT (EXISTS (SELECT 1 
FROM address 
WHERE user_account.id = address.user_id))
2025-04-22 17:07:17,100 INFO sqlalchemy.engine.Engine [generated in 0.00136s] ()


[('Patrick McStar',), ('Squidward Tentacles',), ('Eugene H. Krabs',)]

In [229]:
stmt = select(Address.email_address).where(Address.user.has(User.name == "sandy"))
session.execute(stmt).all()

2025-04-22 17:07:17,114 INFO sqlalchemy.engine.Engine SELECT address.email_address 
FROM address 
WHERE EXISTS (SELECT 1 
FROM user_account 
WHERE user_account.id = address.user_id AND user_account.name = ?)
2025-04-22 17:07:17,115 INFO sqlalchemy.engine.Engine [generated in 0.00116s] ('sandy',)


[('sandy@sqlalchemy.org',), ('sandy@squirrelpower.org',)]

## Loader Strategies


### Selectin Load


In [230]:
from sqlalchemy.orm import selectinload

stmt = select(User).options(selectinload(User.addresses)).order_by(User.id)
for row in session.execute(stmt):
    print(
        f"{row.User.name}  ({', '.join(a.email_address for a in row.User.addresses)})"
    )

2025-04-22 17:07:17,129 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account ORDER BY user_account.id
2025-04-22 17:07:17,130 INFO sqlalchemy.engine.Engine [generated in 0.00113s] ()
2025-04-22 17:07:17,133 INFO sqlalchemy.engine.Engine SELECT address.user_id AS address_user_id, address.id AS address_id, address.email_address AS address_email_address 
FROM address 
WHERE address.user_id IN (?, ?, ?, ?, ?, ?)
2025-04-22 17:07:17,134 INFO sqlalchemy.engine.Engine [generated in 0.00090s] (1, 2, 3, 4, 5, 6)
spongebob  (spongebob@sqlalchemy.org)
sandy  (sandy@sqlalchemy.org, sandy@squirrelpower.org)
patrick  ()
squidward  ()
ehkrabs  ()
pkrabs  (peral.krabs@gmail.com, peral@aol.com)


### Joined Load


In [231]:
from sqlalchemy.orm import joinedload

stmt = (
    select(Address)
    .options(joinedload(Address.user, innerjoin=True))
    .order_by(Address.id)
)
for row in session.execute(stmt):
    print(f"{row.Address.email_address} {row.Address.user.name}")

2025-04-22 17:07:17,148 INFO sqlalchemy.engine.Engine SELECT address.id, address.email_address, address.user_id, user_account_1.id AS id_1, user_account_1.name, user_account_1.fullname 
FROM address JOIN user_account AS user_account_1 ON user_account_1.id = address.user_id ORDER BY address.id
2025-04-22 17:07:17,149 INFO sqlalchemy.engine.Engine [generated in 0.00123s] ()
spongebob@sqlalchemy.org spongebob
sandy@sqlalchemy.org sandy
sandy@squirrelpower.org sandy
peral.krabs@gmail.com pkrabs
peral@aol.com pkrabs


### Explicit Join + Eager load


In [232]:
from sqlalchemy.orm import contains_eager

stmt = (
    select(Address)
    .join(Address.user)
    .where(User.name == "pkrabs")
    .options(contains_eager(Address.user))
    .order_by(Address.id)
)
for row in session.execute(stmt):
    print(f"{row.Address.email_address} {row.Address.user.name}")

2025-04-22 17:07:17,162 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname, address.id AS id_1, address.email_address, address.user_id 
FROM address JOIN user_account ON user_account.id = address.user_id 
WHERE user_account.name = ? ORDER BY address.id
2025-04-22 17:07:17,164 INFO sqlalchemy.engine.Engine [generated in 0.00200s] ('pkrabs',)
peral.krabs@gmail.com pkrabs
peral@aol.com pkrabs


In [233]:
stmt = (
    select(Address)
    .join(Address.user)
    .where(User.name == "pkrabs")
    .options(joinedload(Address.user))
    .order_by(Address.id)
)
print(stmt)  # SELECT has a JOIN and LEFT OUTER JOIN unnecessarily

SELECT address.id, address.email_address, address.user_id, user_account_1.id AS id_1, user_account_1.name, user_account_1.fullname 
FROM address JOIN user_account ON user_account.id = address.user_id LEFT OUTER JOIN user_account AS user_account_1 ON user_account_1.id = address.user_id 
WHERE user_account.name = :name_1 ORDER BY address.id


### Raiseload


##### mapped_column() derives the datatype and nullability from the Mapped annotation

In [234]:
from typing import Optional

from sqlalchemy.orm import DeclarativeBase
from sqlalchemy.orm import Mapped
from sqlalchemy.orm import mapped_column


class Base(DeclarativeBase):
    pass


class SomeClass(Base):
    __tablename__ = "some_table"

    # primary_key=True, therefore will be NOT NULL
    id: Mapped[int] = mapped_column(primary_key=True)

    # not Optional[], therefore will be NOT NULL
    data: Mapped[str]

    # Optional[], therefore will be NULL
    additional_info: Mapped[Optional[str]]

In [235]:
class SomeClass(Base):
    # ...

    # will be String() NOT NULL, but can be None in Python
    data: Mapped[Optional[str]] = mapped_column(nullable=False)

/tmp/ipykernel_167079/2772472934.py:1: SAWarning: This declarative base already contains a class with the same class name and module name as __main__.SomeClass, and will be replaced in the string-lookup table.
  class SomeClass(Base):


InvalidRequestError: Class <class '__main__.SomeClass'> does not have a __table__ or __tablename__ specified and does not inherit from an existing table-mapped class.

In [ ]:
class SomeClass(Base):
    # ...

    # will be String() NULL, but type checker will not expect
    # the attribute to be None
    data: Mapped[str] = mapped_column(nullable=True)